In [3]:
%load_ext autoreload
%autoreload 2

import os
import polars as pl

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
train_size_snrna = [5439, 2720, 1359, 680]
train_size_scmul = [4560, 2280, 1136, 572]
train_size = {"snrna": train_size_snrna, "scmul": train_size_scmul}
trn_grads = [8, 4, 2, 1]
_map_grad2size_scrna = {
    8: 5439,
    4: 2720,
    2: 1359,
    1: 680
}
_map_grad2size_scmul = {
    8: 4560,
    4: 2280,
    2: 1136,
    1: 572
}
_map_grad2size = {"snrna": _map_grad2size_scrna, "scmul": _map_grad2size_scmul}

In [5]:
df_example = pl.read_csv("/home/wuch/prjs/git_nwafu/DeepTAN/src/metrics.clus.scGPT.scmul.csv")
df_example_colnames = df_example.columns
print(df_example_colnames)

['method', 'dataset', 'seed', 'train_size', 'n_feat', 'kbet', 'asw_true_label', 'asw_pred_label', 'ari', 'nmi', 'ami', 'ari_leiden', 'nmi_leiden', 'ami_leiden']


In [9]:
def format_deeptan_cluster(example_colnames, _dataset, _path):
    dfs_deeptan_clus = pl.read_csv(_path).drop(["path", "Capability", "fname", "seed"]).filter(pl.col("split")=="tst").drop(["split"]).filter(pl.col("task") == "multitask").drop(["task"]).rename({"seed_num": "seed"})
    dfs_deeptan_clus = dfs_deeptan_clus.with_columns(pl.col("trn_grad").replace(_map_grad2size[_dataset]).alias("train_size"))
    dfs_deeptan_clus = pl.DataFrame({"dataset": [_dataset] * len(dfs_deeptan_clus)}).hstack(dfs_deeptan_clus).rename({"Method": "method"}).select(example_colnames)
    dfs_deeptan_clus.write_csv(f"metrics.clus.DeepTAN.{_dataset}.csv")

In [7]:
paths_dfs_deeptan_clus = [i for i in os.listdir("./") if i.startswith("result.") and i.endswith("clustering.csv")]
print(paths_dfs_deeptan_clus)

['result.sc_mul.summary_clustering.csv', 'result.sc_rna.summary_clustering.csv']


In [13]:
_path = paths_dfs_deeptan_clus[0]
if "sc_rna" in _path:
    _dataset = "snrna"
if "sc_mul" in _path:
    _dataset = "scmul"

format_deeptan_cluster(df_example_colnames, _dataset, _path)